# 실습5 — 반도체 이미지 denoising (Colab / A100)

배포된 `train_denoising_example.ipynb` 와 **같은 데이터 · 같은 노이즈 합성 · 같은 지표**를 쓰고
학습 쪽만 손본 파이프라인이다. 코드는 노트북에 박아 넣지 않고
[ds-practice](https://github.com/kithhooni-commits/ds-practice) 저장소의 `실습5/src/denoise/` 를 clone 해서 쓴다.
(로컬에서 돌린 것과 완전히 같은 코드라는 뜻이다.)

바꾼 것과 이유는 `실습5/README.md` 에 정리돼 있다. 요약하면:
median 채널 추가 · Charbonnier loss · cosine LR · 긴 학습 · rot90 증강 · 8× self-ensemble.

**실행 순서**: 런타임을 A100 으로 바꾸고 위에서부터 순서대로.

## 0. 런타임 확인

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())

## 1. Drive 마운트

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 2. 데이터 위치 지정

`DATA_ROOT` 는 `train/`, `val/`, `test_label/`, `test_noise_only/` 를 담고 있는 폴더다.
배포 zip 을 그대로 풀었으면 `test_noise_only/test_noise_only/` 처럼 한 겹 더 들어가 있어도 된다 —
코드가 두 경우를 모두 받는다.

In [ ]:
import os
from pathlib import Path

DATA_ROOT = Path("/content/drive/MyDrive/실습프로젝트/dataset")   # TODO: 본인 경로로
os.environ["DS_DATA"] = str(DATA_ROOT)

for sub in ("train", "val", "test_label", "test_noise_only"):
    p = DATA_ROOT / sub
    n = len(list(p.glob("**/*.npy"))) if p.exists() else 0
    print(f"{'OK ' if n else '없음'} {sub:<18} {n:>5} npy   {p}")

### Drive 가 느리면 로컬 디스크로 복사

Colab 에서 Drive I/O 는 학습 속도의 병목이 되기 쉽다. 7,268장이면 복사가 몇 분 안에 끝나고,
그 뒤로는 epoch 마다 로컬 SSD 에서 읽는다. A100 을 쓸 거면 사실상 필수다.

In [ ]:
USE_LOCAL_COPY = True

if USE_LOCAL_COPY:
    LOCAL = Path("/content/dataset")
    if not LOCAL.exists():
        !cp -r "{DATA_ROOT}" "{LOCAL}"
    DATA_ROOT = LOCAL
    os.environ["DS_DATA"] = str(DATA_ROOT)
    !du -sh "{LOCAL}"
print("DS_DATA =", os.environ["DS_DATA"])

## 3. 코드 받기

In [ ]:
REPO = Path("/content/ds-practice")
if REPO.exists():
    !cd "{REPO}" && git pull --ff-only
else:
    !git clone --depth 1 https://github.com/kithhooni-commits/ds-practice.git "{REPO}"

SRC = REPO / "실습5" / "src" / "denoise"
RUNS = Path("/content/runs")
RUNS.mkdir(exist_ok=True)
!ls "{SRC}"

## 4. 지표 검증 — 먼저 여기부터

우리가 재는 PSNR/SSIM 이 채점 숫자와 같은지 확인한다. 배포된 예시 로그
(`log_denoising_example/00012_train/baseline_metrics.json`)의 mean/median/adaptive 성적을
우리 로더·우리 지표로 다시 재서 대조한다.

**`PSNR 최대 차이: 0.0000 dB — 일치` 가 나와야 다음으로 넘어간다.**
여기가 어긋나면 학습 결과도 믿을 수 없다.

(예시 로그를 Drive 에 안 올렸으면 대조 없이 우리 숫자만 나온다. 그래도 상관없다.)

In [ ]:
LOG_EXAMPLE = Path("/content/drive/MyDrive/실습프로젝트/log_denoising_example")
if LOG_EXAMPLE.exists() and not (REPO / "실습5" / "data" / "log_denoising_example").exists():
    (REPO / "실습5" / "data").mkdir(parents=True, exist_ok=True)
    !cp -r "{LOG_EXAMPLE}" "{REPO}/실습5/data/"

!cd "{SRC}" && python check_baselines.py

## 5. 학습

A100 기준 설정이다. 로컬 6GB GPU 에서는 `--patch 128 --batch 16` 을 썼지만,
A100 이면 **크롭 없이 256² 전체**를 배치 32로 돌릴 수 있다 — test 와 완전히 같은 크기로
배우는 셈이라 크롭보다 유리하다.

| 인자 | A100 | 6GB 로컬 | 비고 |
|---|---|---|---|
| `--patch` | 256 | 128 | 256 이면 크롭이 사실상 없음 |
| `--batch` | 32 | 16 | |
| `--epochs` | 60 | 40 | 대략 40분 / 70분 |
| `--model` | `dncnn_plus` | | `dncnn` 으로 두면 배포 구조 그대로 |

`dncnn_plus` 는 median 3×3 결과를 두 번째 입력 채널로 넣은 것뿐이고 파라미터는 576개만 는다.
salt & pepper 가 유독 어려운 문제(입력 17.3 dB)를 겨냥한 설계다.

In [ ]:
!cd "{SRC}" && python train.py \
    --model dncnn_plus \
    --epochs 60 \
    --patch 256 \
    --batch 32 \
    --lr 3e-4 \
    --loss charbonnier \
    --workers 8 \
    --data "{DATA_ROOT}" \
    --out "{RUNS}" \
    --tag a100

### 같은 레시피로 구조만 원본 — 공정한 ablation

median 채널이 실제로 기여했는지 보려면 나머지 조건을 똑같이 두고 `dncnn` 을 한 번 더 돌린다.
시간이 급하면 건너뛰어도 제출에는 지장 없다.

In [ ]:
!cd "{SRC}" && python train.py \
    --model dncnn \
    --epochs 60 --patch 256 --batch 32 --lr 3e-4 --loss charbonnier --workers 8 \
    --data "{DATA_ROOT}" --out "{RUNS}" --tag a100_ablation

## 6. 평가 — 제출값 산출

`test_noise_only` 를 입력으로 넣고 `test_label` 로 채점한다. conventional 비교군
(mean/median/adaptive)도 같은 데이터로 함께 재서 표로 낸다.

`--self-ensemble` 은 dihedral 8종으로 추론해 되돌려 평균내는 것이다. 학습 비용 0,
추론만 8배 느려지고 (100장이라 몇 초), 보통 0.2~0.4 dB 를 공짜로 준다.

In [ ]:
CKPT = sorted(RUNS.glob("*a100/checkpoints/checkpoint_best.ckpt"))[-1]
print("checkpoint:", CKPT)

# self-ensemble 없이 / 있게 각각 재서 그 차이도 발표 근거로 남긴다
!cd "{SRC}" && python evaluate.py "{CKPT}" --data "{DATA_ROOT}" --figures
print("=" * 70)
!cd "{SRC}" && python evaluate.py "{CKPT}" --data "{DATA_ROOT}" --self-ensemble --figures

### 노이즈 종류별 before/after

In [ ]:
from IPython.display import Image, display
for p in sorted(CKPT.parent.parent.glob("test_grid*.png")):
    print(p.name)
    display(Image(filename=str(p)))

## 7. 제출

마지막 셀이 찍은 `제출값 → PSNR_total ... SSIM_total ...` 두 숫자를
`denoising_challenge_score.xlsx` 에 소수 둘째자리로 적는다.

넘어야 할 기준선은 배포 예시 로그(DnCNN 10 epoch)의 **PSNR 30.51 / SSIM 0.8950** 이다.

In [ ]:
import shutil
OUT = Path("/content/drive/MyDrive/실습프로젝트/runs_mine")
OUT.mkdir(parents=True, exist_ok=True)
run = CKPT.parent.parent
shutil.copytree(run, OUT / run.name, dirs_exist_ok=True)
print("Drive 에 저장:", OUT / run.name)
!ls -la "{OUT}/{run.name}"